# Test Analytical and Semantic Layers

In [ ]:
# --- JUPYTER MAGIC ---
%load_ext autoreload
%autoreload 2

import os
import sys
import sqlite3
import pandas as pd
from pathlib import Path

# Add project root to path
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from src.config import load_settings, root_path

settings = load_settings()
test_db = root_path("data/test_ledger.db")

print("Environment loaded. Ready to test Phase 4 (Knowledge Substrate).")
print(f"Targeting Database: {test_db}")

## Step 1 - Analytical Inventory & Temporal Snapshots

In [ ]:
from src.inventory.build import build_inventory
from src.state.snapshot import create_snapshot, list_snapshots

print("--- 1. BUILDING ANALYTICAL INVENTORY ---")
test_inventory_path = root_path("artifacts/test_inventory.parquet")

# Build the Parquet file from the Production Inventory table
df = build_inventory(db_path=test_db, output=test_inventory_path, table="production_inventory")

print(f"✅ Parquet Inventory built at {test_inventory_path}")
print(f"Total Records: {len(df)}")
display(df.head())

print("\n--- 2. CREATING TEMPORAL SNAPSHOT ---")
test_snapshot_dir = root_path("artifacts/test_snapshots")
snapshot_path = create_snapshot(df, test_snapshot_dir)

print(f"✅ Snapshot created at: {snapshot_path}")

# List all snapshots to prove it worked
snapshots = list_snapshots(test_snapshot_dir)
print(f"Total Snapshots stored: {len(snapshots)}")

## Step 2 - The OKF Semantic Wiki

In [ ]:
from src.okf.builder import OKFBuilder

print("--- 3. GENERATING OKF MARKDOWN WIKI ---")
test_okf_dir = root_path("knowledge/test_okf")

# Run the OKF Builder
builder = OKFBuilder(db_path=test_db, okf_dir=test_okf_dir)
builder.build()

# Let's read the generated Master Index to verify it!
index_file = test_okf_dir / "index.md"
if index_file.exists():
    print("\n📄 Contents of OKF index.md:")
    print("="*50)
    print(index_file.read_text(encoding="utf-8"))
    print("="*50)
else:
    print("❌ OKF Index generation failed.")

## Step 3 - The RDF Knowledge Graph

In [ ]:
from src.kg.builder import build_rdf_graph

print("--- 4. BUILDING RDF KNOWLEDGE GRAPH ---")
test_ttl_path = root_path("knowledge/kg/export/test_graph.ttl")

# Build the RDF Graph
graph = build_rdf_graph(db_path=test_db, output_ttl=test_ttl_path)

print(f"\n✅ Knowledge Graph built with {len(graph)} triples.")
print(f"Exported to: {test_ttl_path}")

# Let's inspect the first 10 triples to see the raw Semantic Web data!
print("\n🔍 Inspecting Graph Triples (First 10):")
print("="*50)
for i, (subject, predicate, obj) in enumerate(graph):
    if i >= 10: break
    
    # Clean up the URIs for easier reading in the notebook
    s = str(subject).split('/')[-1]
    p = str(predicate).split('/')[-1].split('#')[-1]
    o = str(obj).split('/')[-1] if 'http' in str(obj) else str(obj)
    
    print(f"[{s}] --({p})--> [{o}]")
print("="*50)

## The Production Facade Test

In [ ]:
from src.pipeline.knowledge import run_knowledge_pipeline

print("--- 5. TESTING THE PRODUCTION FACADE ---")
# We use the mock_nextcloud directory we created in Notebook 1
mock_nextcloud = root_path("notebooks/mock_nextcloud")

# This triggers the exact function your Prefect Flow and Streamlit Dashboard will use
result = run_knowledge_pipeline(db_path=test_db, nextcloud_mount=mock_nextcloud)

print(f"\n✅ Facade Execution Status: {result['status']}")
print("If you check your Qdrant and Fuseki instances, they have now been updated with the test data!")